# Início

In [1]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import scipy
from scipy.interpolate import PchipInterpolator, interp1d
from scipy import signal
from scipy.stats import pearsonr
from scipy.signal import butter, filtfilt
from scipy.signal import find_peaks
import pickle
from scipy.signal.windows import dpss
from scipy.fft import rfft, rfftfreq
from sklearn.feature_selection import mutual_info_regression

In [2]:
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs 
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

In [3]:
def picotar(x, J, passo):
    N = x.shape[0]
    Nj = (N-J)//passo + 1
    X = np.zeros((Nj, J))
    for i in range(Nj):
        X[i,:] = x[(i*passo):(i*passo+J)]
    return X

In [4]:
def scipy_interpolate_signal(series, method='pchip'):
    y = series.values
    x = np.arange(len(y))
    
    # Identifica índices válidos (não-NaN)
    valid_mask = ~np.isnan(y)
    
    # Se todos forem NaN ou houver apenas 1 ponto válido, retorna como está
    if valid_mask.sum() < 2:
        return series
        
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]
    
    if method == 'pchip':
        # PCHIP do SciPy (com suporte a extrapolação nas pontas)
        interpolator = PchipInterpolator(x_valid, y_valid, extrapolate=True)
        y_filled = interpolator(x)
    elif method == 'cubic':
        # Spline cúbico com interpolação linear/extrapolação de segurança nas pontas
        interpolator = interp1d(x_valid, y_valid, kind='cubic', fill_value='extrapolate', bounds_error=False)
        y_filled = interpolator(x)
    else:
        interpolator = interp1d(x_valid, y_valid, kind='linear', fill_value='extrapolate', bounds_error=False)
        y_filled = interpolator(x)
        
    return pd.Series(y_filled, index=series.index)

In [5]:
actis = ['climbingdown', 'climbingup', 'jumping', 'lying', 'running', 'sitting', 'standing', 'walking']
posis = ['chest', 'forearm', 'head', 'shin', 'thigh', 'upperarm', 'waist']
users = ['proband' + x for x in np.arange(1,16).astype(str)]
data = np.load('C:\\Meu Drive\\Doutorado Unicamp\\Projeto\\github\\chang-UDA-HAR\\notebooks\\Xycort.npz')
Xcort = data['Xcort'] * 20 # scale factor to convert to m/s^2
ycort = data['ycort']
J = 150
fs = 50
data = np.load('C:\\Meu Drive\\Doutorado Unicamp\\Projeto\\github\\chang-UDA-HAR\\notebooks\\f0est.npz')
f0est = data['f0est']
Vm = np.mean(Xcort, axis=1)
Xm = Xcort - np.mean(Xcort, axis=1, keepdims=True)
En = np.sum(Xm*Xm, axis=(1,2)).reshape(-1, 1)

In [6]:
# Faixa de busca do f0 no RealWorld
faixa = {(0, 0): [0.5, 1.2],
         (0, 1): [0.5, 1.2],
         (0, 2): [1.5, 2.1],
         (0, 4): [0.9, 1.7],
         (0, 7): [0.6, 1.2],
         (1, 0): [0.6, 1.2],
         (1, 1): [0.5, 1.2],
         (1, 2): [1.5, 2.5],
         (1, 4): [0.9, 1.6],
         (1, 7): [0.7, 1.2],
         (2, 0): [0.5, 1.2],
         (2, 1): [0.5, 1.2],
         (2, 2): [1.5, 2.1],
         (2, 4): [0.9, 1.7],
         (2, 7): [0.7, 1.2],
         (3, 0): [0.5, 1.2],
         (3, 1): [0.5, 1.2],
         (3, 2): [1.5, 2.2],
         (3, 4): [0.9, 1.7],
         (3, 7): [0.7, 1.2],
         (4, 0): [0.5, 1.2],
         (4, 1): [0.6, 1.2],
         (4, 2): [1.7, 2.3],
         (4, 4): [0.9, 1.7],
         (4, 7): [0.7, 1.2],
         (5, 0): [0.6, 1.2],
         (5, 1): [0.6, 1.2],
         (5, 2): [1.7, 2.3],
         (5, 4): [0.9, 1.6],
         (5, 7): [0.6, 1.1],
         (6, 0): [0.6, 1.3],
         (6, 1): [0.6, 1.3],
         (6, 7): [0.6, 1.],
         (6, 4): [0.9, 1.7],
         (6, 2): [1.5, 2.3],
         (7, 0): [0.6, 1.3],
         (7, 1): [0.3, 1.1],
         (7, 7): [0.6, 1.2],
         (7, 4): [0.9, 1.6],
         (7, 2): [1.5, 2.3],
         (8, 0): [0.6, 1.2],
         (8, 1): [0.6, 1.2],
         (8, 7): [0.6, 1.2],
         (8, 4): [0.9, 1.5],
         (8, 2): [1.4, 2.3],
         (9, 0): [0.6, 1.3],
         (9, 1): [0.6, 1.3],
         (9, 7): [0.7, 1.3],
         (9, 4): [0.9, 1.6],
         (9, 2): [1.5, 2.3],
         (10, 0): [0.6, 1.3],
         (10, 1): [0.5, 1.2],
         (10, 7): [0.6, 1.2],
         (10, 4): [0.9, 1.5],
         (10, 2): [1.5, 2.5],
         (11, 0): [0.6, 1.2],
         (11, 1): [0.5, 1.2],
         (11, 7): [0.7, 1.2],
         (11, 4): [0.9, 1.5],
         (11, 2): [1.5, 2.5],
         (12, 0): [0.6, 1.3],
         (12, 1): [0.5, 1.2],
         (12, 7): [0.6, 1.3],
         (12, 4): [0.9, 1.52],
         (12, 2): [1.5, 2.5],
         (13, 0): [0.5, 1.3],
         (13, 1): [0.6, 1.3],
         (13, 7): [0.6, 1.3],
         (13, 4): [0.9, 1.7],
         (13, 2): [1.5, 2.2],
         (14, 0): [0.5, 1.3],
         (14, 1): [0.6, 1.2],
         (14, 7): [0.6, 1.2],
         (14, 4): [0.9, 1.6],
         (14, 2): [1.5, 2.7]}

In [7]:
df_RW = pd.read_csv('df_RW2.csv', index_col=False)
df_sto = pd.read_csv('df_sto2.csv', index_col=False)
df_trc = pd.read_csv('df_trc2.csv', index_col=False)

In [8]:
# Agora vamos visualizar os dados do RealWorld, do arquivo .sto e do arquivo .trc para comparar as acelerações dos sensores
dom = 'shin'  # Podemos mudar para 'chest', 'forearm', 'head', 'thigh', 'shin', 'upperarm' ou 'waist' para comparar outros sensores
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_X'], mode='lines', name=f'RealWorld {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Y'], mode='lines', name=f'RealWorld {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Z'], mode='lines', name=f'RealWorld {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_X'], mode='lines', name=f'STO {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Y'], mode='lines', name=f'STO {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Z'], mode='lines', name=f'STO {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_X'], mode='lines', name=f'TRC {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Y'], mode='lines', name=f'TRC {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Z'], mode='lines', name=f'TRC {dom.capitalize()} Z'))
fig.update_layout(title=f'Comparação das Acelerações do Sensor {dom.capitalize()} entre RW, STO e TRC', xaxis_title='Tempo (s)', yaxis_title='Aceleração')
fig.show()

In [9]:
# 1. Definir os sufixos das 17 variáveis de cada sensor IMU
imu_features = [
    'temp', 
    'acc16_x', 'acc16_y', 'acc16_z',  # Aceleração (escala ±16g)
    'acc6_x', 'acc6_y', 'acc6_z',     # Aceleração (escala ±6g)
    'gyro_x', 'gyro_y', 'gyro_z',     # Giroscópio
    'mag_x', 'mag_y', 'mag_z',        # Magnetômetro
    'orient_x', 'orient_y', 'orient_z', 'orient_w' # Orientação (Quatérnios)
]

# 2. Gerar a lista exata de 54 colunas
columns = ['timestamp', 'activity_id', 'heart_rate']
for sensor_location in ['hand', 'chest', 'ankle']:
    for feature in imu_features:
        columns.append(f'IMU_{sensor_location}_{feature}')

# 3. Ler o arquivo (ajuste o caminho se necessário)
file_path = 'C://Meu Drive//Doutorado Unicamp//Projeto//Dataset//PAMAP2_Dataset//Protocol//subject101.dat'

# Usa expressão regular r'\s+' porque os dados podem ter múltiplos espaços
df = pd.read_csv(file_path, sep=r'\s+', header=None, names=columns)

# # 4. Interpolação das colunas de aceleração para cada atividade
acc_cols = [
    'IMU_ankle_acc16_x', 'IMU_ankle_acc16_y', 'IMU_ankle_acc16_z',
    'IMU_hand_acc16_x', 'IMU_hand_acc16_y', 'IMU_hand_acc16_z',
    'IMU_chest_acc16_x', 'IMU_chest_acc16_y', 'IMU_chest_acc16_z']

# Aplicando no seu DataFrame agrupado por blocos de atividade:
df['activity_block'] = (df['activity_id'] != df['activity_id'].shift()).cumsum()

for col in acc_cols:
    df[col] = df.groupby('activity_block')[col].transform(
        lambda group: scipy_interpolate_signal(group, method='pchip')
    )

df = df.drop(columns=['activity_block'])

actisP = ['lying', 'sitting', 'standing', 'walking', 'running', 'cycling', 
          'Nordic', 'watching TV', 'computer', 'car driving', 'ascending stairs', 'descending stairs', 
          'vacuum', 'ironing', 'folding', 'house cleaning', 'soccer', 'rope jumping']
activity_id = [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 24]

In [10]:
inds = df['activity_id'] == 5
fig = go.Figure()
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_hand_acc16_x'], mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_hand_acc16_y'], mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_hand_acc16_z'], mode='lines', name='Eixo Z'))
fig.update_layout(xaxis_title='Tempo (s)', yaxis_title='Aceleração (g)', legend_title='Componentes', title='Aceleração da Mão durante a corrida')
fig.show()

In [11]:
inds = df['activity_id'] == 5
fig = go.Figure()
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_chest_acc16_x'], mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_chest_acc16_y'], mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_chest_acc16_z'], mode='lines', name='Eixo Z'))
fig.update_layout(xaxis_title='Tempo (s)', yaxis_title='Aceleração (g)', legend_title='Componentes', title='Aceleração do Torso durante a corrida')
fig.show()

In [12]:
inds = df['activity_id'] == 5
fig = go.Figure()
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_x'], mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_y'], mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_z'], mode='lines', name='Eixo Z'))
fig.update_layout(xaxis_title='Tempo (s)', yaxis_title='Aceleração (g)', legend_title='Componentes', title='Aceleração do Pé durante a corrida')
fig.show()

# Figuras no Domínio do tempo

In [14]:
# 1. Filtrar os dados de corrida (activity_id == 5)
df_run = df[df['activity_id'] == 5]

# 2. Criar a estrutura com 3 subplots empilhados
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
colors = {'x': '#1f77b4', 'y': '#ff7f0e', 'z': '#2ca02c'}

ini = 11040
fim = 11440
for sensor_name, row in sensors:
    for axis in ['x', 'y', 'z']:
        fig.add_trace(
            go.Scatter(
                x=df_run['timestamp'].iloc[ini:fim] - df_run['timestamp'].iloc[ini],
                y=df_run[f'IMU_{sensor_name}_acc16_{axis}'].iloc[ini:fim],
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8)
            ),
            row=row, col=1
        )

# 4. Rótulos dos eixos e layout final
# for r in range(1, 4):
#     fig.update_yaxes(title_text='Aceleração (m/s²)', row=r, col=1)

fig.update_yaxes(title_text='Sensor na Mão (m/s²)', row=1, col=1)
fig.update_yaxes(title_text='Sensor no Peito (m/s²)', row=2, col=1)
fig.update_yaxes(title_text='Sensor no Tornozelo (m/s²)', row=3, col=1)

fig.update_xaxes(title_text='Tempo (s)', row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.84,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [71]:
fig.write_image("fig1.pdf")

In [48]:
k = 1 # 9.81 / 1912
for sensor_name, row in sensors:
    x = df_run[f'IMU_{sensor_name}_acc16_{'x'}'].iloc[ini:fim] * k
    y = df_run[f'IMU_{sensor_name}_acc16_{'y'}'].iloc[ini:fim] * k
    z = df_run[f'IMU_{sensor_name}_acc16_{'z'}'].iloc[ini:fim] * k
    sig = np.array([x, y, z])
    acc_g = np.sqrt(np.sum(sig.mean(axis=1) ** 2))
    print(f"Aceleração média do sensor {sensor_name}: {acc_g:.2f} g")
    fig = px.line(sig.T)
    fig.show()

Aceleração média do sensor hand: 12.34 g


Aceleração média do sensor chest: 10.17 g


Aceleração média do sensor ankle: 16.16 g


In [28]:
k = 9.81 / 1912
for sensor_name in ['chest', 'forearm', 'head', 'thigh', 'shin', 'upperarm', 'waist']:
    x = df_sto[f'{sensor_name}_{'X'}'].iloc[ini:fim] * k
    y = df_sto[f'{sensor_name}_{'Y'}'].iloc[ini:fim] * k
    z = df_sto[f'{sensor_name}_{'Z'}'].iloc[ini:fim] * k
    sig = np.array([x, y, z])
    acc_g = np.sqrt(np.sum(sig.mean(axis=1) ** 2))
    print(f"Aceleração média do sensor {sensor_name}: {acc_g:.2f} g")
    fig = px.line(sig.T)
    fig.show()

Aceleração média do sensor chest: 9.81 g


Aceleração média do sensor forearm: 1.99 g


Aceleração média do sensor head: 9.81 g


Aceleração média do sensor thigh: 15.17 g


Aceleração média do sensor shin: 9.87 g


Aceleração média do sensor upperarm: 7.92 g


Aceleração média do sensor waist: 11.41 g


In [62]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('forearm', 1), ('chest', 2), ('shin', 3)]
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}

ini = 1
fim = 200
k = 9.81 / 1912
for sensor_name, row in sensors:
    for axis in ['X', 'Y', 'Z']:
        fig.add_trace(
            go.Scatter(
                x=df_sto['time'].iloc[ini:fim],
                y=df_sto[f'{sensor_name}_{axis}'].iloc[ini:fim] * k,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8)
            ),
            row=row, col=1
        )

fig.update_yaxes(title_text='Sensor na Mão (m/s²)', row=1, col=1)
fig.update_yaxes(title_text='Sensor no Peito (m/s²)', row=2, col=1)
fig.update_yaxes(title_text='Sensor no Tornozelo (m/s²)', row=3, col=1)

fig.update_xaxes(title_text='Tempo (s)', row=3, col=1)

# fig.update_layout(
#     height=800,
#     width=900,
#     # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
#     legend_title_text='Componentes',
#     showlegend=True
# )

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.84,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [63]:
fig.write_image("fig1.pdf")

In [30]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('forearm', 1), ('chest', 2), ('shin', 3)]
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}

ini = 1
fim = 200
for sensor_name, row in sensors:
    for axis in ['X', 'Y', 'Z']:
        fig.add_trace(
            go.Scatter(
                x=df_trc['time'].iloc[ini:fim],
                y=df_trc[f'{sensor_name}_{axis}'].iloc[ini:fim]/1000,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8)
            ),
            row=row, col=1
        )

fig.update_yaxes(title_text='Sensor na Mão (m/s²)', row=1, col=1)
fig.update_yaxes(title_text='Sensor no Peito (m/s²)', row=2, col=1)
fig.update_yaxes(title_text='Sensor no Tornozelo (m/s²)', row=3, col=1)

fig.update_xaxes(title_text='Tempo (s)', row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=False
)

fig.show()

In [32]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('forearm', 1), ('chest', 2), ('shin', 3)]
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}

ini = 1
fim = 200
for sensor_name, row in sensors:
    for axis in ['X', 'Y', 'Z']:
        fig.add_trace(
            go.Scatter(
                x=df_RW['time'].iloc[ini:fim],
                y=df_RW[f'{sensor_name}_{axis}'].iloc[ini:fim],
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8)
            ),
            row=row, col=1
        )

fig.update_yaxes(title_text='Sensor na Mão (m/s²)', row=1, col=1)
fig.update_yaxes(title_text='Sensor no Peito (m/s²)', row=2, col=1)
fig.update_yaxes(title_text='Sensor no Tornozelo (m/s²)', row=3, col=1)

fig.update_xaxes(title_text='Tempo (s)', row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=False
)

fig.show()

# Figuras no Domínio da frequência

In [13]:
dom = 'shin'
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}
J = 150
window = signal.windows.hann(J*2)
dim = 9
fs = 50
vals = np.linspace(0, fs/2, 500)
Spec_final = np.zeros(1500)
A_spec_final = np.zeros_like(vals)
fig = go.Figure()
for eixo in ['X', 'Y', 'Z']:
    s = df_sto[f'{dom}_{eixo}'].values[:J*2]
    s = s - s.mean()
    s = s * window
    X = picotar(s, J, 1)
    Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
    R = Xm.T.dot(Xm)
    l, v = np.linalg.eig(R)
    B = v[:,dim:]
    Pmu = []
    for f in vals:
        e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
        Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
    A = 20*np.log10(np.array(Pmu))+40
    aux = np.hstack((s, np.zeros(2700)))
    Xf = np.fft.fft(aux)
    Spec_final += abs(Xf[:1500])
    A_spec_final += np.array(Pmu)
    freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
    cor = np.correlate(s, s, 'full')[-2*J:]
    peaks, _ = find_peaks(cor, height=0)
    f0 = fs/peaks[cor[peaks].argmax()]
    ind = f0/fs*3000
    inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
    smoothed = butter_lowpass_filter(20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), cutoff=2.5, fs=60)
    fig.add_trace(go.Scatter(x=freq[:1500], y=smoothed,
                             line=dict(color=colors[eixo], width=1.8),
                             mode="lines",
                             showlegend=True, 
                             name='fft '+eixo))
peaks, _ = find_peaks(Spec_final, height=0)
# f0 = freq[peaks[Spec_final[peaks].argmax()]]
f0 = freq[peaks[3]]
fig.update_layout(
    width=900,
    height=450,
    # showlegend=False,
    xaxis=dict(
        showgrid=True,
        range=[0, 10],
        tickvals= np.arange(1,(fs/2)/f0)*f0,
        ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2),
    )
)
A_spec_final = 20*np.log10(A_spec_final) - 20*np.log10(A_spec_final.min()) - 20
fig.add_trace(go.Scatter(x=vals,y=A_spec_final, mode="lines", showlegend=True, name='MUSIC', line=dict(color='magenta', width=1.8)))
# fig.add_trace(go.Scatter(x=freq[:1500],y=20*np.log10(Spec_final/np.sqrt(300)+1e-3), mode="lines", showlegend=True, name='fft soma dos eixos'))
fig.update_yaxes(title_text='Amplitude (dB)')
fig.update_xaxes(title_text='Frequência (Hz)')
fig.show()

In [74]:
fig.write_image("fig1.pdf")

In [68]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

sensors = [('forearm', 1), ('chest', 2), ('shin', 3)]
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}
fs = 50
J = 150
dim = 9
window = signal.windows.hann(J*2)
k = 9.81 / 1912

for sensor_name, row in sensors:
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for axis in ['X', 'Y', 'Z']:
        s = df_sto[f'{sensor_name}_{axis}'].values[:J*2] * k
        s = s - s.mean()
        s = s * window
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))+40
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
        smoothed = butter_lowpass_filter(20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), cutoff=2.5, fs=60)
        fig.add_trace(
            go.Scatter(
                x=freq[:1500], y=smoothed,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8),
            ),
            row=row, col=1
        )
    A_spec_final = 20*np.log10(A_spec_final) - 20
    fig.add_trace(
        go.Scatter(
            x=vals, y=A_spec_final,
            mode='lines',
            name='Pseudoespectro MUSIC',
            legendgroup='MUSIC',
            showlegend=(row == 1),
            line=dict(color='magenta', width=1.8),
        ),
        row=row, col=1
    )

fig.update_yaxes(title_text='Amplitude na Mão (dB)', row=1, col=1)
fig.update_yaxes(title_text='Amplitude no Peito (dB)', row=2, col=1)
fig.update_yaxes(title_text='Amplitude no Tornozelo (dB)', row=3, col=1)
peaks, _ = find_peaks(Spec_final, height=0)
f0 = freq[peaks[Spec_final[peaks].argmax()]]
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=1, col=1)
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=2, col=1)
fig.update_xaxes(title_text='Frequência (Hz)',
                 tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.74,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [ ]:
df_run = df[df['activity_id'] == 5]
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
colors = {'x': '#1f77b4', 'y': '#ff7f0e', 'z': '#2ca02c'}

fs = 50
ini = 11041
fim = 11640
for sensor_name, row in sensors:
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for axis in ['x', 'y', 'z']:
        s = df_run[f'IMU_{sensor_name}_acc16_{axis}'].iloc[ini:fim]
        s = scipy.signal.resample(s, 300)
        s = s - s.mean()
        s = s * window
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))+40
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
        smoothed = butter_lowpass_filter(20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), cutoff=2.5, fs=60)
        fig.add_trace(
            go.Scatter(
                x=freq[:1500], y=smoothed,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8),
            ),
            row=row, col=1
        )
    A_spec_final = 20*np.log10(A_spec_final) - 20*np.log10(A_spec_final.min()) - 40
    fig.add_trace(
        go.Scatter(
            x=vals, y=A_spec_final,
            mode='lines',
            name='Pseudoespectro MUSIC',
            legendgroup='MUSIC',
            showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
            line=dict(color='magenta', width=1.8),
        ),
        row=row, col=1
    )

fig.update_yaxes(title_text='Amplitude na Mão (dB)', row=1, col=1)
fig.update_yaxes(title_text='Amplitude no Peito (dB)', row=2, col=1)
fig.update_yaxes(title_text='Amplitude no Tornozelo (dB)', row=3, col=1)
peaks, _ = find_peaks(Spec_final, height=0)
f0 = freq[peaks[Spec_final[peaks].argmax()]]
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=1, col=1)
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=2, col=1)
fig.update_xaxes(title_text='Frequência (Hz)',
                 tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.74,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

# Modelo Harmônico

## f0

In [11]:
# ini = 10040
# fim = 10640
# ini = 11040
# fim = 11640
ini = 0
fim = 300
dom = 'hand'
sensor_name = 'forearm'
# x = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_x'].iloc[ini:fim].values, 300)
# y = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_y'].iloc[ini:fim].values, 300)
# z = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_z'].iloc[ini:fim].values, 300)
x = df_sto[f'{sensor_name}_X'].iloc[ini:fim].values
y = df_sto[f'{sensor_name}_Y'].iloc[ini:fim].values
z = df_sto[f'{sensor_name}_Z'].iloc[ini:fim].values
x = x - x.mean()
y = y - y.mean()
z = z - z.mean()
fs = 50
px.line(np.array([x, y, z]).T)

In [12]:
J = 150
fmax = 25
f0s = np.arange(1, 3.1, 0.001)
X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
erro = []
razao = []
for f0 in f0s:
    L = np.floor(fmax/f0).astype(int)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(X)
    recon = theta.dot(coefs)
    erro.append(np.sqrt(np.mean((X - recon)**2)))
    razao.append(np.sum(recon**2)/np.sum(X**2))
fig = px.line(x=np.sort(f0s), y=razao,
        title=f'Erro de Reconstrução para Frequências Fundamentais Candidatas do Sensor {dom.capitalize()}', 
        labels={'x': 'Frequência Fundamental (Hz)', 'y': 'Erro de Reconstrução'}).show()
f0 = f0s[np.argmin(erro)]
print(f'Frequência fundamental estimada: {f0:.3f} Hz')

Frequência fundamental estimada: 1.443 Hz


In [244]:
act = -1
df_run = df[df['activity_id'] == activity_id[act]]
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
sens_pt = ['Mão', 'Peito', 'Tornozelo']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# ini = 11040
# fim = 11640
ini = 2040
fim = 2640
fmax = 25
L = 4
f0s = np.arange(0.82, 2.46, 0.001)
fig = go.Figure()
for sensor_name, row in sensors:
    x = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_x'].iloc[ini:fim].values, 300)
    y = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_y'].iloc[ini:fim].values, 300)
    z = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_z'].iloc[ini:fim].values, 300)
    x = x - x.mean()
    y = y - y.mean()
    z = z - z.mean()
    X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
    erro = []
    razao = []
    for f0 in f0s:
        # L = np.floor(fmax/f0).astype(int)
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(X)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.mean((X - recon)**2)))
        razao.append(np.sum(recon**2)/np.sum(X**2))
    fig.add_trace(
        go.Scatter(
            x=f0s,
            y=razao,
            mode='lines',
            name=f'{sens_pt[row-1]}',
            line=dict(color=colors[row-1], width=2.1)
        )
    )

fig.update_yaxes(title_text='Ajuste do Modelo Harmônico', range=[0, 1])
fig.update_xaxes(title_text='Frequência Fundamental (Hz)')
f0 = 1.065
fig.update_layout(
    height=450,
    width=900,
    title_text='Busca da Frequência Fundamental para Diferentes Posições do Sensor em '+actisP[act],
    legend_title_text='Posições do Sensor',
    showlegend=True,
    xaxis=dict(
        showgrid=True,
        tickvals= np.arange(1,(3.1)/f0)*f0,
        ticktext= (np.arange(1,(3.1)/f0)*f0).round(3),
    ),
    legend=dict(
        x=0.38,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [245]:
fig.write_image("fig1.pdf")

In [54]:
f0 = 1.326
fmax = 25
L = np.floor(fmax/f0).astype(int)
J = 150
arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
C = np.cos(arg)
S = np.sin(arg)
theta = np.hstack((C, S))
X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
coefs = np.linalg.pinv(theta).dot(X)
recon = theta.dot(coefs)
erro_recon = np.sqrt(np.mean((X - recon)**2, axis=1))
t = np.arange(J)/fs
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=X[:, 0], mode='lines', name='Sinal Original x'))
fig.add_trace(go.Scatter(x=t, y=X[:, 1], mode='lines', name='Sinal Original y'))
fig.add_trace(go.Scatter(x=t, y=X[:, 2], mode='lines', name='Sinal Original z'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 0], mode='lines', name='Sinal Reconstruído x'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 1], mode='lines', name='Sinal Reconstruído y'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 2], mode='lines', name='Sinal Reconstruído z'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 0] - X[:, 0], mode='lines', name='Resíduo x'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 1] - X[:, 1], mode='lines', name='Resíduo y'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 2] - X[:, 2], mode='lines', name='Resíduo z'))
fig.add_trace(go.Scatter(x=t, y=erro_recon, mode='lines', name='Erro de Reconstrução', line=dict(dash='dash'), line_color='black'))
fig.update_layout(title=f'Reconstrução do Sinal {dom.capitalize()}', xaxis_title='Tempo (s)', yaxis_title='Valor')
fig.show()

In [55]:
R = np.sqrt(coefs[:L,:]**2 + coefs[L:,:]**2)
hAmp = np.sqrt(np.sum(R**2, axis=1))
hAmp

array([15.64677222,  5.79016337,  7.93580196,  8.95135757,  5.00809252,
        6.0563099 ,  5.93479393,  5.66026375,  5.486826  ,  5.11535001,
        4.6629432 ,  4.37227983,  4.46804138,  4.24439741,  4.33847283,
        3.65187659,  3.20814143,  3.46997556])

In [56]:
fig = go.Figure()
for i in range(3):
    Sx = np.fft.fft(X[:, i], n=1024)
    Sr = np.fft.fft(recon[:, i], n=1024)
    Se = np.fft.fft(X[:, i] - recon[:, i], n=1024)
    freq = np.fft.fftfreq(1024, d=1/fs)
    fig.add_trace(go.Scatter(x=freq[:512], y=np.abs(Sx[:512])/J, mode='lines', name=f'FFT Original Eixo {["X", "Y", "Z"][i]}'))
    fig.add_trace(go.Scatter(x=freq[:512], y=np.abs(Sr[:512])/J, mode='lines', name=f'FFT Reconstruída Eixo {["X", "Y", "Z"][i]}'))
    fig.add_trace(go.Scatter(x=freq[:512], y=np.abs(Se[:512])/J, mode='lines', name=f'FFT Resíduo Eixo {["X", "Y", "Z"][i]}'))
fig.update_layout(title=f'FFT do Sinal Original, Reconstruído e Resíduo {dom.capitalize()}', xaxis_title='Frequência (Hz)', yaxis_title='Magnitude')
grid = np.arange(1, L+1) * f0
fig.add_trace(go.Scatter(x=grid, y=hAmp/2, mode='markers', name='Harmônicos', marker=dict(color='black', size=8, symbol='x')))
fig.show()

In [66]:
df_run = df[df['activity_id'] == 5]
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
sens_pt = ['Mão', 'Peito', 'Tornozelo']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

ini = 11040
fim = 11640
f0 = 1.336
L = np.floor(fmax/f0).astype(int)
grid = np.arange(0, L+1) * f0
fig = go.Figure()
for sensor_name, row in sensors:
    x = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_x'].iloc[ini:fim].values, 300)
    y = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_y'].iloc[ini:fim].values, 300)
    z = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_z'].iloc[ini:fim].values, 300)
    x = x - x.mean()
    y = y - y.mean()
    z = z - z.mean()
    X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(X)
    R = np.sqrt(coefs[:L,:]**2 + coefs[L:,:]**2)
    hAmp = np.sqrt(np.sum(R**2, axis=1))
    fig.add_trace(
        go.Scatter(
            x=grid,
            y=np.hstack((0, hAmp)), 
            mode='lines+markers', 
            name=sens_pt[row-1], 
            line=dict(color=colors[row-1], width=2.1)
        )
    )

fig.update_yaxes(title_text='Magnitude')
fig.update_xaxes(title_text='Frequência (Hz)', range=[0, 10])
fig.update_layout(
    height=450,
    width=900,
    title_text='Contorno Espectral Harmônico para Diferentes Posições do Sensor',
    legend_title_text='Posições do Sensor',
    showlegend=True,
    xaxis=dict(
        showgrid=True,
        tickvals= np.arange(1,(fs/2)/f0)*f0,
        ticktext= (np.arange(1,(fs/2)/f0)*f0).round(3),
    ),
    legend=dict(
        x=0.78,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

## Parâmetros

In [16]:
x = df_run['IMU_chest_acc16_x'].values
y = df_run['IMU_chest_acc16_y'].values
z = df_run['IMU_chest_acc16_z'].values
sig = np.array([scipy.signal.resample(x, x.shape[0]//2), scipy.signal.resample(y, y.shape[0]//2), scipy.signal.resample(z, z.shape[0]//2)]).T
for i in range(3):
    b, a = signal.butter(5, 0.3, 'hp', fs=fs)
    zi = signal.lfilter_zi(b, a)
    z, _ = signal.lfilter(b, a, sig[:,i], zi=zi*sig[0,i])
    sig[:,i] = z
X1 = picotar(sig[:, 0], J, 50)
X2 = picotar(sig[:, 1], J, 50)
X3 = picotar(sig[:, 2], J, 50)
X = np.stack((X1, X2, X3), axis=2)

In [17]:
px.line(sig)

In [18]:
f0s = np.arange(1.1, 2.8, 0.01)
f0est = []
for i in tqdm(range(X.shape[0])):
    janela = X[i,:,:]
    erro = []
    for f0 in f0s:
        L = 8
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.mean((janela - recon)**2)))
    f0est.append(f0s[np.argmin(erro)])
f0est = np.array(f0est)

100%|██████████| 244/244 [00:05<00:00, 45.20it/s]


In [19]:
px.line(f0est)

In [38]:
L = 4
hAmps = []
for i, janela in tqdm(enumerate(Xm)):
    f0 = f0est[i]
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    hAmps.append(np.sqrt((coefs[:L,:]**2 + coefs[L:,:]**2).sum(axis=1)))
hAmps = np.array(hAmps)

144647it [00:10, 13719.20it/s]


In [42]:
actis

['climbingdown',
 'climbingup',
 'jumping',
 'lying',
 'running',
 'sitting',
 'standing',
 'walking']

In [44]:
fig = go.Figure()
act1 = 4
act2 = 0
for p in range(7):
    inds = (ycort[:,1]==act1) & (ycort[:,0]==p)
    fig.add_trace(go.Scatter(x=hAmps[inds,0], y=hAmps[inds,1], mode='markers', name=actis[act1] + ' '+ posis[p]))
    inds = (ycort[:,1]==act2) & (ycort[:,0]==p)
    fig.add_trace(go.Scatter(x=hAmps[inds,0], y=hAmps[inds,1], mode='markers', name=actis[act2] + ' '+ posis[p]))
fig.update_layout(width=900, height=550)
fig.show()

## Figuras de Lissajous

In [144]:
J = 150
L = 16
ini = 11040
fim = 12040
f0s = np.arange(1.2, 1.7, 0.001)
dom = 'hand'
xd1 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_x'].iloc[ini:fim].values, 500)
yd1 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_y'].iloc[ini:fim].values, 500)
zd1 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_z'].iloc[ini:fim].values, 500)
dom = 'ankle'
xd2 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_x'].iloc[ini:fim].values, 500)
yd2 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_y'].iloc[ini:fim].values, 500)
zd2 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_z'].iloc[ini:fim].values, 500)
dom = 'chest'
xd3 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_x'].iloc[ini:fim].values, 500)
yd3 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_y'].iloc[ini:fim].values, 500)
zd3 = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_z'].iloc[ini:fim].values, 500)
Xd1 = picotar(xd1 - np.mean(xd1), J, 10)
Yd1 = picotar(yd1 - np.mean(yd1), J, 10)
Zd1 = picotar(zd1 - np.mean(zd1), J, 10)
D1 = np.stack((Xd1, Yd1, Zd1), axis=2)
Xd2 = picotar(xd2 - np.mean(xd2), J, 10)
Yd2 = picotar(yd2 - np.mean(yd2), J, 10)
Zd2 = picotar(zd2 - np.mean(zd2), J, 10)
D2 = np.stack((Xd2, Yd2, Zd2), axis=2)
Xd3 = picotar(xd3 - np.mean(xd3), J, 10)
Yd3 = picotar(yd3 - np.mean(yd3), J, 10)
Zd3 = picotar(zd3 - np.mean(zd3), J, 10)
D3 = np.stack((Xd3, Yd3, Zd3), axis=2)

f0est = []
feats = []
for i in tqdm(range(D1.shape[0])):
    janela = D1[i,:,:]
    erro = []
    for f0 in f0s:
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.sum((janela - recon)**2)))
    f0 = f0s[np.argmin(erro)]
    f0est.append(f0)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    R = np.sqrt(coefs[:L,:]**2 + coefs[L:,:]**2)
    hAmp = np.sqrt(np.sum(R**2, axis=1))
    # feats.append(hAmp)
    feats.append(coefs)
F0d1 = np.array(f0est)
Ampsd1 = np.array(feats)

f0est = []
feats = []
for i in tqdm(range(D2.shape[0])):
    janela = D2[i,:,:]
    erro = []
    for f0 in f0s:
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.sum((janela - recon)**2)))
    f0 = f0s[np.argmin(erro)]
    f0est.append(f0)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    R = np.sqrt(coefs[:L,:]**2 + coefs[L:,:]**2)
    hAmp = np.sqrt(np.sum(R**2, axis=1))
    # feats.append(hAmp)
    feats.append(coefs)
F0d2 = np.array(f0est)
Ampsd2 = np.array(feats)

f0est = []
feats = []
for i in tqdm(range(D3.shape[0])):
    janela = D3[i,:,:]
    erro = []
    for f0 in f0s:
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.sum((janela - recon)**2)))
    f0 = f0s[np.argmin(erro)]
    f0est.append(f0)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    R = np.sqrt(coefs[:L,:]**2 + coefs[L:,:]**2)
    hAmp = np.sqrt(np.sum(R**2, axis=1))
    # feats.append(hAmp)
    feats.append(coefs)
F0d3 = np.array(f0est)
Ampsd3 = np.array(feats)

100%|██████████| 36/36 [00:10<00:00,  3.39it/s]


In [145]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        y=F0d1,
        mode='lines'
    )
)
fig.add_trace(
    go.Scatter(
        y=F0d2,
        mode='lines'
    )
)
fig.add_trace(
    go.Scatter(
        y=F0d3,
        mode='lines'
    )
)
fig.show()

In [92]:
coefs = Ampsd1[10,:,:]
janela = D1[10,:,:]

# Calcula todos os recon para descobrir os limites globais
recons = []
for l in range(L):
    aux = np.zeros_like(coefs)
    aux[l,:], aux[L+l,:] = coefs[l,:], coefs[L+l,:]
    recon = theta.dot(aux)
    recons.append(recon)

# Empilha todos os pontos
all_points = np.vstack(recons)

# Centro dos dados
center = all_points.mean(axis=0)

# Maior extensão entre os três eixos
max_range = np.max(all_points.max(axis=0) - all_points.min(axis=0)) / 2

# Limites iguais para os três eixos
x_lim = [center[0] - max_range, center[0] + max_range]
y_lim = [center[1] - max_range, center[1] + max_range]
z_lim = [center[2] - max_range, center[2] + max_range]

fig = go.Figure()

for l, recon in enumerate(recons, start=1):
    fig.add_trace(go.Scatter3d(
        x=recon[:,0],
        y=recon[:,1],
        z=recon[:,2],
        mode='markers',
        name=f'Harmônico {l}',
        marker=dict(size=5)
    ))

fig.update_layout(
    width=800,
    height=600,
    title='Contribuição de cada Harmônico para a Reconstrução',
    scene=dict(
        xaxis=dict(title='Eixo X', range=x_lim),
        yaxis=dict(title='Eixo Y', range=y_lim),
        zaxis=dict(title='Eixo Z', range=z_lim),
        aspectmode='cube'  # força mesma escala visual
    )
)

fig.show()

In [101]:
coefs = Ampsd2[10,:,:]
janela = D2[10,:,:]

# Calcula todos os recon para descobrir os limites globais
recons = []
for l in range(L):
    aux = np.zeros_like(coefs)
    aux[l,:], aux[L+l,:] = coefs[l,:], coefs[L+l,:]
    recon = theta.dot(aux)
    recons.append(recon)

# Empilha todos os pontos
all_points = np.vstack(recons)

# Centro dos dados
center = all_points.mean(axis=0)

# Maior extensão entre os três eixos
max_range = np.max(all_points.max(axis=0) - all_points.min(axis=0)) / 2

# Limites iguais para os três eixos
x_lim = [center[0] - max_range, center[0] + max_range]
y_lim = [center[1] - max_range, center[1] + max_range]
z_lim = [center[2] - max_range, center[2] + max_range]

fig = go.Figure()

for l, recon in enumerate(recons, start=1):
    fig.add_trace(go.Scatter3d(
        x=recon[:,0],
        y=recon[:,1],
        z=recon[:,2],
        mode='markers',
        name=f'Harmônico {l}',
        marker=dict(size=5)
    ))

fig.update_layout(
    width=800,
    height=600,
    title='Contribuição de cada Harmônico para a Reconstrução',
    scene=dict(
        xaxis=dict(title='Eixo X', range=x_lim),
        yaxis=dict(title='Eixo Y', range=y_lim),
        zaxis=dict(title='Eixo Z', range=z_lim),
        aspectmode='cube'  # força mesma escala visual
    )
)

fig.show()

In [162]:
lm = 6
coefs = Ampsd1[10,:,:]
janela = D1[10,:,:]
recon = theta.dot(coefs)
recons = []
for l in range(lm):
    aux = np.zeros_like(coefs)
    aux[l,:], aux[L+l,:] = coefs[l,:], coefs[L+l,:]
    recon = theta.dot(aux)
    recons.append(recon)
recon = np.array(recons).sum(axis=0)
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=recon[:,0],
    y=recon[:,1],
    z=recon[:,2]+15,
    mode='lines',
    line=dict(width=8)
    # marker=dict(size=15)
))
coefs = Ampsd2[10,:,:]
janela = D2[10,:,:]
recon = theta.dot(coefs)
recons = []
for l in range(lm):
    aux = np.zeros_like(coefs)
    aux[l,:], aux[L+l,:] = coefs[l,:], coefs[L+l,:]
    recon = theta.dot(aux)
    recons.append(recon)
recon = np.array(recons).sum(axis=0)
fig.add_trace(go.Scatter3d(
    x=recon[:,0],
    y=recon[:,1],
    z=recon[:,2],
    mode='lines',
    line=dict(width=5)
))
coefs = Ampsd3[10,:,:]
janela = D3[10,:,:]
recon = theta.dot(coefs)
recons = []
for l in range(lm):
    aux = np.zeros_like(coefs)
    aux[l,:], aux[L+l,:] = coefs[l,:], coefs[L+l,:]
    recon = theta.dot(aux)
    recons.append(recon)
recon = np.array(recons).sum(axis=0)
fig.add_trace(go.Scatter3d(
    x=recon[:,0],
    y=recon[:,1]+20,
    z=recon[:,2]+10,
    mode='lines',
    line=dict(width=5)
))
fig.update_layout(width=800, height=700)
fig.update_layout(scene=dict(aspectmode='cube'))
fig.show()

In [137]:
coefs = Ampsd2[10,:,:]
janela = D2[10,:,:]
recon = theta.dot(coefs)
recon = recons[0] + recons[1] + recons[2] + recons[3] + recons[4] + recons[5] + recons[6] + recons[7]
# recon = scipy.signal.resample_poly(recon, 4, 1)
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=recon[:,0],
    y=recon[:,1],
    z=recon[:,2],
    mode='lines',
    line=dict(width=5)
))
fig.add_trace(go.Scatter3d(
    x=janela[:,0],
    y=janela[:,1],
    z=janela[:,2],
    mode='lines',
    marker=dict(size=5)
))
fig.update_layout(width=800, height=700)
fig.update_layout(scene=dict(aspectmode='cube'))
fig.show()

# Análise de compartilhamento

## Mocap Hamner

## RealWorld

### Inicio

In [119]:
u = 3
act = 1
ses = []
for p in [1, 0, 3]:
    inds = (ycort[:,1]==act) & (ycort[:,2]==u) & (ycort[:,0]==p)
    ses.append(Xcort[inds].reshape(-1,3))
ses = np.concatenate(ses, axis=1)
px.line(ses, title=actis[act])

In [122]:
J = 150
L = 8
ini = 0
fim = len(ses)
f0s = np.arange(faixa[u,act][0], faixa[u,act][1], 0.01)
b, a = signal.butter(5, 0.3, 'hp', fs=fs)
zi = signal.lfilter_zi(b, a)
F0 = []
Feats = []
for dom in range(3):
    x = ses[ini:fim, dom*3]
    x, _ = signal.lfilter(b, a, x, zi=zi*x[0])
    y = ses[ini:fim, dom*3+1]
    y, _ = signal.lfilter(b, a, y, zi=zi*y[0])
    z = ses[ini:fim, dom*3+2]
    z, _ = signal.lfilter(b, a, z, zi=zi*z[0])
    X = picotar(x, J, 150)
    Y = picotar(y, J, 150)
    Z = picotar(z, J, 150)
    D = np.stack((X, Y, Z), axis=2)
    f0est = []
    feats = []
    for i in tqdm(range(D.shape[0])):
        janela = D[i,:,:]
        erro = []
        razao = []
        for f0 in f0s:
            # L = np.floor(24/f0).astype(int)
            arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
            C = np.cos(arg)
            S = np.sin(arg)
            theta = np.hstack((C, S))
            coefs = np.linalg.pinv(theta).dot(janela)
            recon = theta.dot(coefs)
            erro.append(np.sqrt(np.mean((janela - recon)**2)))
            razao.append(np.sum(recon**2)/np.sum(janela**2))
        # f0 = f0s[np.argmin(erro)]
        f0 = f0s[np.argmax(razao)]
        f0est.append(f0)
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        feats.append(coefs)
    F0.append(np.array(f0est))
    Feats.append(np.array(feats))
px.line(np.array(F0).T)

100%|██████████| 92/92 [00:00<00:00, 92.69it/s]


In [260]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=x, mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(y=y, mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(y=z, mode='lines', name='Eixo Z'))
fig.show()

In [123]:
r, p_pearson = pearsonr(F0[1], F0[0])
print(f'Mão x Peito cor: {r.round(2)} p: {p_pearson.round(3)}')
r, p_pearson = pearsonr(F0[0], F0[2])
print(f'Mão x Torno cor: {r.round(2)} p: {p_pearson.round(3)}')
r, p_pearson = pearsonr(F0[1], F0[2])
print(f'Tor x Peito cor: {r.round(2)} p: {p_pearson.round(3)}')

Mão x Peito cor: 0.36 p: 0.0
Mão x Torno cor: 0.3 p: 0.004
Tor x Peito cor: 0.7 p: 0.0


In [326]:
act = 7
inds = (ycort[:,1]==act) & (ycort[:,2]==0)
px.histogram(data['f0est'][inds])

In [327]:
act = 7
inds = (ycort[:,1]==act) & (ycort[:,2]==3)
px.line(data['f0est'][inds])

In [227]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
map = [1, 0, 3]
colors = {'x': '#1f77b4', 'y': '#ff7f0e', 'z': '#2ca02c'}

fs = 50
window = signal.windows.hann(J*2)
dim = 8
ini = 2000
fim = 2300
for sensor_name, row in sensors:
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for axis in range(3):
        s = ses[ini:fim, map[row-1]*3+axis]
        s = s - s.mean()
        s = s * window
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))+40
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
        smoothed = butter_lowpass_filter(20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), cutoff=2.5, fs=60)
        fig.add_trace(
            go.Scatter(
                x=freq[:1500], y=smoothed,
                mode='lines',
                name=f'Eixo {list(colors.keys())[axis].upper()}',
                legendgroup=list(colors.keys())[axis],        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[list(colors.keys())[axis]], width=1.8),
            ),
            row=row, col=1
        )
    A_spec_final = 20*np.log10(A_spec_final) - 20*np.log10(A_spec_final.min()) - 40
    fig.add_trace(
        go.Scatter(
            x=vals, y=A_spec_final,
            mode='lines',
            name='Pseudoespectro MUSIC',
            legendgroup='MUSIC',
            showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
            line=dict(color='magenta', width=1.8),
        ),
        row=row, col=1
    )

fig.update_yaxes(title_text='Magnitude na Mão (dB)', row=1, col=1)
fig.update_yaxes(title_text='Magnitude no Peito (dB)', row=2, col=1)
fig.update_yaxes(title_text='Magnitude no Tornozelo (dB)', row=3, col=1)
peaks, _ = find_peaks(Spec_final, height=0)
f0 = freq[peaks[Spec_final[peaks].argmax()]]
# f0 = 1.05
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=1, col=1)
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=2, col=1)
fig.update_xaxes(title_text='Frequência (Hz)',
                 tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.74,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [247]:
fig.write_image("fig1.pdf")

In [246]:
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
map = [1, 0, 3]
sens_pt = ['Mão', 'Peito', 'Tornozelo']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

ini = 2000
fim = 2300
fmax = 25
L = 4
f0s = np.arange(0.82, 2.46, 0.001)
fig = go.Figure()
for sensor_name, row in sensors:
    x = ses[ini:fim, map[row-1]*3+0]
    y = ses[ini:fim, map[row-1]*3+1]
    z = ses[ini:fim, map[row-1]*3+2]
    x = x - x.mean()
    y = y - y.mean()
    z = z - z.mean()
    X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
    erro = []
    razao = []
    for f0 in f0s:
        # L = np.floor(fmax/f0).astype(int)
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(X)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.mean((X - recon)**2)))
        razao.append(np.sum(recon**2)/np.sum(X**2))
    fig.add_trace(
        go.Scatter(
            x=f0s,
            y=razao,
            mode='lines',
            name=f'{sens_pt[row-1]}',
            line=dict(color=colors[row-1], width=2.1)
        )
    )

fig.update_yaxes(title_text='Ajuste do Modelo Harmônico', range=[0, 1])
fig.update_xaxes(title_text='Frequência Fundamental (Hz)')
f0 = 1.122
fig.update_layout(
    height=450,
    width=900,
    title_text='Busca da Frequência Fundamental para Diferentes Posições do Sensor em '+actis[act],
    legend_title_text='Posições do Sensor',
    showlegend=True,
    xaxis=dict(
        showgrid=True,
        tickvals= np.arange(1,(3.1)/f0)*f0,
        ticktext= (np.arange(1,(3.1)/f0)*f0).round(3),
    ),
    legend=dict(
        x=0.38,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

### Teste de correlação F0

In [133]:
u = 7
act = 1
ses = []
for p in [1, 0, 3]:
    inds = (ycort[:,1]==act) & (ycort[:,2]==u) & (ycort[:,0]==p)
    ses.append(Xcort[inds].reshape(-1,3))
ses = np.concatenate(ses, axis=1)
px.line(ses, title=f'{actis[act]} {users[u]}')

In [134]:
J = 150
L = 8
ini = 0
fim = len(ses)
f0s = np.arange(faixa[u,act][0], faixa[u,act][1], 0.01)
b, a = signal.butter(5, 0.3, 'hp', fs=fs)
zi = signal.lfilter_zi(b, a)
F0 = []
Feats = []
for dom in range(3):
    x = ses[ini:fim, dom*3]
    x, _ = signal.lfilter(b, a, x, zi=zi*x[0])
    y = ses[ini:fim, dom*3+1]
    y, _ = signal.lfilter(b, a, y, zi=zi*y[0])
    z = ses[ini:fim, dom*3+2]
    z, _ = signal.lfilter(b, a, z, zi=zi*z[0])
    X = picotar(x, J, 150)
    Y = picotar(y, J, 150)
    Z = picotar(z, J, 150)
    D = np.stack((X, Y, Z), axis=2)
    f0est = []
    feats = []
    for i in tqdm(range(D.shape[0])):
        janela = D[i,:,:]
        erro = []
        razao = []
        for f0 in f0s:
            # L = np.floor(24/f0).astype(int)
            arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
            C = np.cos(arg)
            S = np.sin(arg)
            theta = np.hstack((C, S))
            coefs = np.linalg.pinv(theta).dot(janela)
            recon = theta.dot(coefs)
            erro.append(np.sqrt(np.mean((janela - recon)**2)))
            razao.append(np.sum(recon**2)/np.sum(janela**2))
        # f0 = f0s[np.argmin(erro)]
        f0 = f0s[np.argmax(razao)]
        f0est.append(f0)
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        feats.append(coefs)
    F0.append(np.array(f0est))
    Feats.append(np.array(feats))
px.line(np.array(F0).T)

100%|██████████| 248/248 [00:02<00:00, 94.57it/s]


In [135]:
r, p_pearson = pearsonr(F0[1], F0[0])
print(f'Mão x Peito cor: {r.round(2)} p: {p_pearson.round(3)}')
r, p_pearson = pearsonr(F0[0], F0[2])
print(f'Mão x Torno cor: {r.round(2)} p: {p_pearson.round(3)}')
r, p_pearson = pearsonr(F0[1], F0[2])
print(f'Tor x Peito cor: {r.round(2)} p: {p_pearson.round(3)}')

Mão x Peito cor: 0.35 p: 0.0
Mão x Torno cor: 0.32 p: 0.0
Tor x Peito cor: 0.48 p: 0.0


### Regiões de f0 constante

In [376]:
fig = go.Figure()
for u in range(7):
    for act in [4, 7]:
        inds = (ycort[:,1]==act) & (ycort[:,2]==u)
        perf = data['f0est'][inds]
        # print(perf.shape[0]/7)
        perf = perf.reshape(perf.shape[0]//7, -1, order='F').mean(axis=1)
        med = perf.mean().round(3)
        des = (perf.std()/med*100).round(1)        
        fig.add_trace(go.Scatter(x=np.arange(len(perf)), y=perf.ravel(), mode='lines', name=f'{u} {actis[act]} {med}-{des}'))
fig.show()

In [ ]:
# atividades e usuarios com perfil de f0 estável (desvio padrão de até 3% do valor médio)
# [[4, 1], [7, 1], [4, 2], [7, 2], [7, 3], [7, 4], [4, 5], [4, 6],]

### Compartilhamento

In [201]:
act = 7
u = 3
ses = []
for p in range(7):
    inds = (ycort[:,1]==act) & (ycort[:,2]==u) & (ycort[:,0]==p)
    ses.append(Xcort[inds].reshape(-1,3))
ses = np.concatenate(ses, axis=1)
px.line(ses, title=actis[act])

In [ ]:
# atividades e usuarios com perfil de f0 estável (desvio padrão de até 3% do valor médio)
# [[4, 1], [7, 1], [4, 2], [7, 2], [7, 3], [7, 4], [4, 5], [4, 6],]

In [257]:
J = 150
L = 8
ini = 0
fim = ses.shape[0]
b, a = signal.butter(5, 0.3, 'hp', fs=fs)
zi = signal.lfilter_zi(b, a)
if act == 3: f0s = np.arange(0.71, 1.15, 0.005) # Caminhada
elif act == 4: f0s = np.arange(1.05, 1.55, 0.005) # Corrida
F0 = []
Feats = []
D = []
for dom in [1, 0, 3]:
    x = ses[ini:fim, dom*3]
    x, _ = signal.lfilter(b, a, x, zi=zi*x[0])
    y = ses[ini:fim, dom*3+1]
    y, _ = signal.lfilter(b, a, y, zi=zi*y[0])
    z = ses[ini:fim, dom*3+2]
    z, _ = signal.lfilter(b, a, z, zi=zi*z[0])
    X = picotar(x, J, 150)
    Y = picotar(y, J, 150)
    Z = picotar(z, J, 150)
    D.append(np.stack((X, Y, Z), axis=2))
D = np.concatenate(D, axis=-1)
f0est = []
feats = []
for i in tqdm(range(D.shape[0])):
    janela = D[i,:,:]
    erro = []
    for f0 in f0s:
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.mean((janela - recon)**2)))
    f0 = f0s[np.argmin(erro)]
    f0est.append(f0)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    feats.append(coefs)
F0 = np.array(f0est)
Feats = np.array(feats)

  0%|          | 0/201 [00:00<?, ?it/s]

100%|██████████| 201/201 [00:03<00:00, 60.98it/s]


In [203]:
px.line(F0)

In [109]:
RW_feats = {}

In [239]:
with open('RW_feats.pkl', 'rb') as f:
    RW_feats = pickle.load(f)
print(RW_feats.keys())

dict_keys(['proband2 walking'])


In [110]:
RW_feats[users[u]+' '+actis[act]] = [F0, Feats]
print(RW_feats.keys())

dict_keys(['proband2 walking'])


In [111]:
with open('RW_feats.pkl', 'wb') as f:
    pickle.dump(RW_feats, f)

In [258]:
Desc = []
for d in range(3):
    desc = []
    for i in range(Feats.shape[0]):
        coefs = Feats[i,:,d*3:(d+1)*3]
        aux = []
        for l in range(L):
            A = coefs[l,:]
            B = coefs[l+L,:]
            M = np.vstack((A, B)).T
            G = M.T.dot(M)
            w, v = np.linalg.eigh(G)
            w = np.sort(w)[::-1]
            sige = np.sqrt(w)
            El = np.sqrt(w.sum())
            e = np.sqrt(1 - (w[1]/w[0]))
            aux.append([El, e, sige[0], sige[1]])
        desc.append(aux)
    Desc.append(np.array(desc))
Desc = np.array(Desc)

In [259]:
fig = go.Figure()
for d1 in range(3):
    for l1 in range(4):
        aux1 = Desc[d1,:,l1,0] - Desc[d1,:,l1,0].mean()
        corr = scipy.signal.correlate(aux1, aux1, mode='full')
        fig.add_trace(go.Scatter(y=corr[len(corr)//2:]/corr.max(), mode='lines', name=f'D {d1+1} H {l1+1}x{l1+1}'))
fig.show()

In [ ]:
fig = go.Figure()
for d1 in range(3):
    for d2 in range(d1+1,3):
        for l1 in range(4):
            for l2 in range(l1,4):
                aux1 = Desc[d1,:,l1,0] - Desc[d1,:,l1,0].mean()
                aux2 = Desc[d2,:,l2,0] - Desc[d2,:,l2,0].mean()
                corr = scipy.signal.correlate(aux1, aux2, mode='full')
                fig.add_trace(go.Scatter(y=corr[len(corr)//2:]/corr.max(), mode='lines', name=f'D{d1+1}x{d2+1} H{l1+1}x{l2+1}'))
fig.show()

In [235]:
for d1 in range(3):
    for d2 in range(d1+1,3):
        for l1 in range(4):
            for l2 in range(l1,4):
                aux1 = Desc[d1,:,l1,0] - Desc[d1,:,l1,0].mean()
                aux2 = Desc[d2,:,l2,0] - Desc[d2,:,l2,0].mean()
                r, p_pearson = pearsonr(aux1, aux2)
                corr = scipy.signal.correlate(aux1, aux2, mode='valid')
                print(f'D {d1+1}x{d2+1} H {l1+1}x{l2+1}: {r.round(2)} p: {p_pearson.round(3)}')

D 1x2 H 1x1: 0.12 p: 0.087
D 1x2 H 1x2: 0.31 p: 0.0
D 1x2 H 1x3: 0.12 p: 0.082
D 1x2 H 1x4: 0.03 p: 0.639
D 1x2 H 2x2: 0.47 p: 0.0
D 1x2 H 2x3: 0.22 p: 0.002
D 1x2 H 2x4: 0.21 p: 0.003
D 1x2 H 3x3: 0.03 p: 0.662
D 1x2 H 3x4: -0.2 p: 0.004
D 1x2 H 4x4: 0.09 p: 0.202
D 1x3 H 1x1: 0.32 p: 0.0
D 1x3 H 1x2: 0.23 p: 0.001
D 1x3 H 1x3: 0.29 p: 0.0
D 1x3 H 1x4: 0.07 p: 0.301
D 1x3 H 2x2: 0.18 p: 0.009
D 1x3 H 2x3: 0.27 p: 0.0
D 1x3 H 2x4: 0.08 p: 0.245
D 1x3 H 3x3: 0.04 p: 0.6
D 1x3 H 3x4: -0.05 p: 0.466
D 1x3 H 4x4: 0.13 p: 0.064
D 2x3 H 1x1: 0.19 p: 0.007
D 2x3 H 1x2: 0.2 p: 0.004
D 2x3 H 1x3: 0.08 p: 0.268
D 2x3 H 1x4: -0.17 p: 0.014
D 2x3 H 2x2: 0.28 p: 0.0
D 2x3 H 2x3: 0.59 p: 0.0
D 2x3 H 2x4: 0.03 p: 0.638
D 2x3 H 3x3: 0.35 p: 0.0
D 2x3 H 3x4: 0.03 p: 0.669
D 2x3 H 4x4: 0.24 p: 0.0


In [243]:
px.line(Desc[1,:,:,0])

In [219]:
fig = go.Figure()
for l1 in range(8):
    for l2 in range(l1,8):
        fig.add_trace(go.Scatter(x=Desc[0,:,l1,0], y=Desc[1,:,l2,0], mode='markers', name=f'{l1}x{l2}'))
fig.update_layout(width=700)
fig.show()

In [207]:
posisP = ['Mão', 'Peito', 'Tornozelo']
Qui, Pv, rotulo = [], [], []
for d1 in range(3):
    for d2 in range(d1+1, 3):
        for l1 in range(L):
            for l2 in range(L):
                s1 = np.sign(Desc[d1,1:,l1,0] - Desc[d1,:-1,l1,0])
                s2 = np.sign(Desc[d2,1:,l2,0] - Desc[d2,:-1,l2,0])
                tC = np.zeros((2,2), dtype=int)
                tC[0,0] = np.sum(s1==1)
                tC[0,1] = np.sum(s1==-1)
                tC[1,0] = np.sum(s2==1)
                tC[1,1] = np.sum(s2==-1)
                tE = np.zeros((2,2))
                for i in range(2):
                    for j in range(2):
                        tE[i,j] = tC[i,:].sum() * tC[:,j].sum() / tC.sum()
                qui2 = ((tC - tE)**2 / tE).sum()
                pValor = 1 - scipy.stats.chi2.cdf(qui2, df=1)
                Qui.append(qui2)
                Pv.append(pValor)
                rotulo.append(f'P {d1} x {d2} - H {l1+1} x {l2+1}')

In [208]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=Qui, mode='markers', name='Qui-quadrado'))
fig.add_trace(go.Scatter(y=Pv, mode='markers', name='p-valor'))
fig.update_layout(title='Teste de Qui-quadrado para comparação entre domínios', xaxis_title='Par de domínios comparados', yaxis_title='Valor do teste', height=700)
fig.update_xaxes(tickmode='array', tickvals=np.arange(len(rotulo)), ticktext=rotulo)
fig.show()

In [209]:
np.array(rotulo)[np.array(Pv)<0.05]

array([], dtype='<U17')

## PAMAP2

### Inicio

In [280]:
act = 5
inds = df['activity_id'] == activity_id[act]
fig = go.Figure()
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_x'], mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_y'], mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_z'], mode='lines', name='Eixo Z'))
fig.update_layout(xaxis_title='Tempo (s)', yaxis_title='Aceleração (g)', legend_title='Componentes', title='Aceleração durante '+actisP[act])
fig.show()

In [271]:
df_asp = df[df['activity_id'] == activity_id[act]]
fig = go.Figure()
fig.add_trace(go.Scatter(y=df_asp['IMU_hand_acc16_x'], name='Mão X'))
fig.add_trace(go.Scatter(y=df_asp['IMU_hand_acc16_y'], name='Mão Y'))
fig.add_trace(go.Scatter(y=df_asp['IMU_hand_acc16_z'], name='Mão Z'))
fig.add_trace(go.Scatter(y=df_asp['IMU_chest_acc16_x'], name='Peito X'))
fig.add_trace(go.Scatter(y=df_asp['IMU_chest_acc16_y'], name='Peito Y'))
fig.add_trace(go.Scatter(y=df_asp['IMU_chest_acc16_z'], name='Peito Z'))
fig.add_trace(go.Scatter(y=df_asp['IMU_ankle_acc16_x'], name='Torno X'))
fig.add_trace(go.Scatter(y=df_asp['IMU_ankle_acc16_y'], name='Torno Y'))
fig.add_trace(go.Scatter(y=df_asp['IMU_ankle_acc16_z'], name='Torno Z'))
fig.show()

In [279]:
file_path = 'C://Meu Drive//Doutorado Unicamp//Projeto//Dataset//PAMAP2_Dataset//Protocol//subject101.dat'
df = pd.read_csv(file_path, sep=r'\s+', header=None, names=columns)
df['activity_block'] = (df['activity_id'] != df['activity_id'].shift()).cumsum()
for col in acc_cols:
    df[col] = df.groupby('activity_block')[col].transform(
        lambda group: scipy_interpolate_signal(group, method='pchip')
    )
df = df.drop(columns=['activity_block'])

In [277]:
df_run = df[df['activity_id'] == activity_id[act]]
J = 150
L = 8
ini = 0
fim = df_run.shape[0]
f0s = np.arange(0.53, 1.25, 0.01)
b, a = signal.butter(5, 0.3, 'hp', fs=fs)
zi = signal.lfilter_zi(b, a)
F0 = []
Feats = []
for dom in ['hand', 'chest', 'ankle']:
    x = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_x'].iloc[ini:fim].values, fim // 2)
    x, _ = signal.lfilter(b, a, x, zi=zi*x[0])
    y = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_y'].iloc[ini:fim].values, fim // 2)
    y, _ = signal.lfilter(b, a, y, zi=zi*y[0])
    z = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_z'].iloc[ini:fim].values, fim // 2)
    z, _ = signal.lfilter(b, a, z, zi=zi*z[0])
    X = picotar(x, J, 30)
    Y = picotar(y, J, 30)
    Z = picotar(z, J, 30)
    D = np.stack((X, Y, Z), axis=2)
    f0est = []
    feats = []
    for i in tqdm(range(D.shape[0])):
        janela = D[i,:,:]
        erro = []
        razao = []
        for f0 in f0s:
            # L = np.floor(24/f0).astype(int)
            arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
            C = np.cos(arg)
            S = np.sin(arg)
            theta = np.hstack((C, S))
            coefs = np.linalg.pinv(theta).dot(janela)
            recon = theta.dot(coefs)
            erro.append(np.sqrt(np.mean((janela - recon)**2)))
            razao.append(np.sum(recon**2)/np.sum(janela**2))
        # f0 = f0s[np.argmin(erro)]
        f0 = f0s[np.argmax(razao)]
        f0est.append(f0)
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        feats.append(coefs)
    F0.append(np.array(f0est))
    Feats.append(np.array(feats))

  0%|          | 0/388 [00:00<?, ?it/s]

100%|██████████| 388/388 [00:04<00:00, 91.79it/s] 


In [275]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=x, mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(y=y, mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(y=z, mode='lines', name='Eixo Z'))
fig.show()

In [278]:
px.line(np.array(F0).T)

In [249]:
dadosF0 = {}

In [56]:
dadosF0[actisP[act]] = F0
print(dadosF0.keys())

dict_keys(['walking', 'ascending stairs', 'descending stairs', 'running', 'cycling'])


In [57]:
with open('dadosF0.pkl', 'wb') as f:
    pickle.dump(dadosF0, f)

In [55]:
with open('dadosF0.pkl', 'rb') as f:
    dadosF0 = pickle.load(f)
print(dadosF0.keys())

dict_keys(['walking', 'ascending stairs', 'descending stairs', 'running'])


In [58]:
cor = scipy.stats.pearsonr(F0[0][50:350], F0[1][50:350])[0]
print('Correlação entre Mão e Peito: ', np.round(cor,3))
cor = scipy.stats.pearsonr(F0[0][50:350], F0[2][50:350])[0]
print('Correlação entre Mão e Torno: ', np.round(cor,3))
cor = scipy.stats.pearsonr(F0[1][50:350], F0[2][50:350])[0]
print('Correlação entre Tor e Peito: ', np.round(cor,3))

Correlação entre Mão e Peito:  0.16
Correlação entre Mão e Torno:  0.068
Correlação entre Tor e Peito:  0.295


In [310]:
df_run = df[df['activity_id'] == activity_id[act]]
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
colors = {'x': '#1f77b4', 'y': '#ff7f0e', 'z': '#2ca02c'}

fs = 50
window = signal.windows.hann(J*2)
dim = 8
ini = 2000
fim = 2600
for sensor_name, row in sensors:
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for axis in ['x', 'y', 'z']:
        s = df_run[f'IMU_{sensor_name}_acc16_{axis}'].iloc[ini:fim]
        s = scipy.signal.resample(s, 300)
        s = s - s.mean()
        s = s * window
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))+40
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
        smoothed = butter_lowpass_filter(20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), cutoff=2.5, fs=60)
        fig.add_trace(
            go.Scatter(
                x=freq[:1500], y=smoothed,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8),
            ),
            row=row, col=1
        )
    A_spec_final = 20*np.log10(A_spec_final) - 20*np.log10(A_spec_final.min()) - 40
    fig.add_trace(
        go.Scatter(
            x=vals, y=A_spec_final,
            mode='lines',
            name='Pseudoespectro MUSIC',
            legendgroup='MUSIC',
            showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
            line=dict(color='magenta', width=1.8),
        ),
        row=row, col=1
    )

fig.update_yaxes(title_text='Magnitude na Mão (dB)', row=1, col=1)
fig.update_yaxes(title_text='Magnitude no Peito (dB)', row=2, col=1)
fig.update_yaxes(title_text='Magnitude no Tornozelo (dB)', row=3, col=1)
peaks, _ = find_peaks(Spec_final, height=0)
f0 = freq[peaks[Spec_final[peaks].argmax()]]
f0 = 0.76
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 16], row=1, col=1)
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 16], row=2, col=1)
fig.update_xaxes(title_text='Frequência (Hz)',
                 tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 16], row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.74,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.64,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [299]:
fig.write_image("fig1.pdf")

In [305]:
act = 5
df_run = df[df['activity_id'] == activity_id[act]]
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
sens_pt = ['Mão', 'Peito', 'Tornozelo']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# ini = 11040
# fim = 11640
ini = 2040
fim = 2640
fmax = 25
L = 18
f0s = np.arange(0.52, 2.06, 0.001)
fig = go.Figure()
for sensor_name, row in sensors:
    x = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_x'].iloc[ini:fim].values, 300)
    y = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_y'].iloc[ini:fim].values, 300)
    z = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_z'].iloc[ini:fim].values, 300)
    x = x - x.mean()
    y = y - y.mean()
    z = z - z.mean()
    X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
    erro = []
    razao = []
    for f0 in f0s:
        # L = np.floor(fmax/f0).astype(int)
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(X)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.mean((X - recon)**2)))
        razao.append(np.sum(recon**2)/np.sum(X**2))
    fig.add_trace(
        go.Scatter(
            x=f0s,
            y=razao,
            mode='lines',
            name=f'{sens_pt[row-1]}',
            line=dict(color=colors[row-1], width=2.1)
        )
    )

fig.update_yaxes(title_text='Ajuste do Modelo Harmônico', range=[0, 1])
fig.update_xaxes(title_text='Frequência Fundamental (Hz)')
f0 = 0.8/2
fig.update_layout(
    height=450,
    width=900,
    title_text='Busca da Frequência Fundamental para Diferentes Posições do Sensor em '+actisP[act],
    legend_title_text='Posições do Sensor',
    showlegend=True,
    xaxis=dict(
        showgrid=True,
        tickvals= np.arange(1,(3.1)/f0)*f0,
        ticktext= (np.arange(1,(3.1)/f0)*f0).round(3),
    ),
    legend=dict(
        x=0.38,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

### Perfil de f0

In [ ]:
# atividades e usuarios com perfil de f0 estável (desvio padrão de até 3% do valor médio)
# [[4, 101], [3, 102], [3, 105], [4, 105], [4, 106], [3, 107]]

In [112]:
user = '104'
file_path = f'C://Meu Drive//Doutorado Unicamp//Projeto//Dataset//PAMAP2_Dataset//Protocol//subject{user}.dat'
df = pd.read_csv(file_path, sep=r'\s+', header=None, names=columns)
df['activity_block'] = (df['activity_id'] != df['activity_id'].shift()).cumsum()
for col in acc_cols:
    df[col] = df.groupby('activity_block')[col].transform(
        lambda group: scipy_interpolate_signal(group, method='pchip')
    )
df = df.drop(columns=['activity_block'])

In [115]:
act = 11 # 3: walking, 4: running
df_run = df[df['activity_id'] == activity_id[act]]
J = 150
L = 8
ini = 0
fim = df_run.shape[0]
if act == 3: f0s = np.arange(0.73, 1.15, 0.001) # Caminhada
elif act == 4: f0s = np.arange(1.05, 1.55, 0.001) # Corrida
elif act == 10: f0s = np.arange(0.35, 0.65, 0.001) # Subir escada
elif act == 11: f0s = np.arange(0.35, 0.65, 0.001) # Descer escada
b, a = signal.butter(5, 0.3, 'hp', fs=fs)
zi = signal.lfilter_zi(b, a)
F0 = []
Feats = []
for dom in ['hand', 'chest', 'ankle']:
    x = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_x'].iloc[ini:fim].values, fim // 2)
    x, _ = signal.lfilter(b, a, x, zi=zi*x[0])
    y = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_y'].iloc[ini:fim].values, fim // 2)
    y, _ = signal.lfilter(b, a, y, zi=zi*y[0])
    z = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_z'].iloc[ini:fim].values, fim // 2)
    z, _ = signal.lfilter(b, a, z, zi=zi*z[0])
    X = picotar(x, J, 150)
    Y = picotar(y, J, 150)
    Z = picotar(z, J, 150)
    D = np.stack((X, Y, Z), axis=2)
    f0est = []
    feats = []
    for i in tqdm(range(D.shape[0])):
        janela = D[i,:,:]
        erro = []
        razao = []
        for f0 in f0s:
            # L = np.floor(24/f0).astype(int)
            arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
            C = np.cos(arg)
            S = np.sin(arg)
            theta = np.hstack((C, S))
            coefs = np.linalg.pinv(theta).dot(janela)
            recon = theta.dot(coefs)
            erro.append(np.sqrt(np.mean((janela - recon)**2)))
            razao.append(np.sum(recon**2)/np.sum(janela**2))
        # f0 = f0s[np.argmin(erro)]
        f0 = f0s[np.argmax(razao)]
        f0est.append(f0)
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        feats.append(coefs)
    F0.append(np.array(f0est))
    Feats.append(np.array(feats))
px.line(np.array(F0).T, title=actisP[act])

  0%|          | 0/47 [00:00<?, ?it/s]

100%|██████████| 47/47 [00:01<00:00, 23.52it/s]


In [116]:
# act = 10 # 3: walking, 4: running
# df_run = df[df['activity_id'] == activity_id[act]]
px.line(df_run.values[:,38:41])

In [117]:
r, p_pearson = pearsonr(F0[1], F0[0])
print(f'Mão x Peito cor: {r.round(2)} p: {p_pearson.round(3)}')
r, p_pearson = pearsonr(F0[0], F0[2])
print(f'Mão x Torno cor: {r.round(2)} p: {p_pearson.round(3)}')
r, p_pearson = pearsonr(F0[1], F0[2])
print(f'Tor x Peito cor: {r.round(2)} p: {p_pearson.round(3)}')

Mão x Peito cor: 0.2 p: 0.179
Mão x Torno cor: 0.02 p: 0.907
Tor x Peito cor: 0.46 p: 0.001


In [65]:
actisP, activity_id

(['lying',
  'sitting',
  'standing',
  'walking',
  'running',
  'cycling',
  'Nordic',
  'watching TV',
  'computer',
  'car driving',
  'ascending stairs',
  'descending stairs',
  'vacuum',
  'ironing',
  'folding',
  'house cleaning',
  'soccer',
  'rope jumping'],
 [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 24])

In [ ]:
with open('dadosF0.pkl', 'rb') as f:
    dadosF0 = pickle.load(f)
print(dadosF0.keys())

In [ ]:
dadosF0[user+' '+actisP[act]] = F0
print(dadosF0.keys())

In [433]:
with open('dadosF0.pkl', 'wb') as f:
    pickle.dump(dadosF0, f)

In [450]:
fig = go.Figure()
for i in dadosF0.keys():
    perf = np.array(dadosF0[i]).mean(axis=0)
    n = perf.shape[0]
    ini, fim = int(0.1*n), int(0.9*n)
    perf = perf[ini:fim]
    med = perf.mean().round(3)
    des = (perf.std()/med*100).round(1)        
    fig.add_trace(go.Scatter(x=np.arange(len(perf)), y=perf.ravel(), mode='lines', name=f'{i} {med}+-{des}'))
fig.show()

In [ ]:
# atividades e usuarios com perfil de f0 estável (desvio padrão de até 3% do valor médio)
# [[4, 101], [3, 102], [3, 105], [4, 105], [4, 106], [3, 107]]

### Análise de compartilhamento

In [ ]:
# atividades e usuarios com perfil de f0 estável (desvio padrão de até 3% do valor médio)
# [[4, 101], [3, 102], [3, 105], [4, 105], [4, 106], [3, 107]]

In [13]:
user = '107'
file_path = f'C://Meu Drive//Doutorado Unicamp//Projeto//Dataset//PAMAP2_Dataset//Protocol//subject{user}.dat'
df = pd.read_csv(file_path, sep=r'\s+', header=None, names=columns)
df['activity_block'] = (df['activity_id'] != df['activity_id'].shift()).cumsum()
for col in acc_cols:
    df[col] = df.groupby('activity_block')[col].transform(
        lambda group: scipy_interpolate_signal(group, method='pchip')
    )
df = df.drop(columns=['activity_block'])

In [14]:
act = 3
df_run = df[df['activity_id'] == activity_id[act]]
J = 150
L = 8
ini = 0
fim = df_run.shape[0]
b, a = signal.butter(5, 0.3, 'hp', fs=fs)
zi = signal.lfilter_zi(b, a)
if act == 3: f0s = np.arange(0.73, 1.15, 0.005) # Caminhada
elif act == 4: f0s = np.arange(1.05, 1.55, 0.005) # Corrida
F0 = []
Feats = []
D = []
for dom in ['hand', 'chest', 'ankle']:
    x = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_x'].iloc[ini:fim].values, fim // 2)
    x, _ = signal.lfilter(b, a, x, zi=zi*x[0])
    y = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_y'].iloc[ini:fim].values, fim // 2)
    y, _ = signal.lfilter(b, a, y, zi=zi*y[0])
    z = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_z'].iloc[ini:fim].values, fim // 2)
    z, _ = signal.lfilter(b, a, z, zi=zi*z[0])
    X = picotar(x, J, 150)
    Y = picotar(y, J, 150)
    Z = picotar(z, J, 150)
    D.append(np.stack((X, Y, Z), axis=2))
D = np.concatenate(D, axis=-1)
f0est = []
feats = []
for i in tqdm(range(D.shape[0])):
    janela = D[i,:,:]
    erro = []
    for f0 in f0s:
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.mean((janela - recon)**2)))
    f0 = f0s[np.argmin(erro)]
    f0est.append(f0)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(janela)
    feats.append(coefs)
F0 = np.array(f0est)
Feats = np.array(feats)

100%|██████████| 112/112 [00:03<00:00, 29.24it/s]


In [495]:
PAMAP_feats = {}

In [13]:
with open('PAMAP_feats.pkl', 'rb') as f:
    PAMAP_feats = pickle.load(f)
print(PAMAP_feats.keys())

dict_keys(['101 running', '102 walking', '105 walking', '105 running', '106 running', '107 walking'])


In [523]:
PAMAP_feats[user+' '+actisP[act]] = [F0, Feats]
print(PAMAP_feats.keys())

dict_keys(['101 running', '102 walking', '105 walking', '105 running', '106 running', '107 walking'])


In [ ]:
with open('PAMAP_feats.pkl', 'wb') as f:
    pickle.dump(PAMAP_feats, f)

In [96]:
[F0, Feats] = PAMAP_feats['105 walking']
# px.line(F0)
Feats = Feats[100:1400,:,:]

In [18]:
Desc = []
for d in range(3):
    desc = []
    for i in range(Feats.shape[0]):
        coefs = Feats[i,:,d*3:(d+1)*3]
        aux = []
        for l in range(L):
            A = coefs[l,:]
            B = coefs[l+L,:]
            M = np.vstack((A, B)).T
            G = M.T.dot(M)
            w, v = np.linalg.eigh(G)
            w = np.sort(w)[::-1]
            sige = np.sqrt(w)
            El = np.sqrt(w.sum())
            e = np.sqrt(1 - (w[1]/w[0]))
            aux.append([El, e, sige[0], sige[1]])
        desc.append(aux)
    Desc.append(np.array(desc))
Desc = np.array(Desc)

In [15]:
px.line(F0)

In [266]:
fig = go.Figure()
for d1 in range(3):
    for l1 in range(4):
        aux1 = Desc[d1,:,l1,0] - Desc[d1,:,l1,0].mean()
        corr = scipy.signal.correlate(aux1, aux1, mode='full')
        fig.add_trace(go.Scatter(y=corr[len(corr)//2:]/corr.max(), mode='lines', name=f'D {d1+1} H {l1+1}x{l1+1}'))
fig.show()

In [267]:
fig = go.Figure()
for d1 in range(3):
    for d2 in range(d1+1,3):
        for l1 in range(4):
            for l2 in range(l1,4):
                aux1 = Desc[d1,:,l1,0] - Desc[d1,:,l1,0].mean()
                aux2 = Desc[d2,:,l2,0] - Desc[d2,:,l2,0].mean()
                corr = scipy.signal.correlate(aux1, aux2, mode='full')
                fig.add_trace(go.Scatter(y=corr[len(corr)//2:]/corr.max(), mode='lines', name=f'D{d1+1}x{d2+1} H{l1+1}x{l2+1}'))
fig.show()

In [32]:
for d1 in range(3):
    for d2 in range(d1+1,3):
        for l1 in range(4):
            for l2 in range(l1,4):
                aux1 = Desc[d1,20:80,l1,0] - Desc[d1,20:80,l1,0].mean()
                aux2 = Desc[d2,20:80,l2,0] - Desc[d2,20:80,l2,0].mean()
                r, p_pearson = pearsonr(aux1, aux2)
                print(f'D {d1+1}x{d2+1} H {l1+1}x{l2+1}: {r.round(2)} p: {p_pearson.round(3)}')

D 1x2 H 1x1: 0.08 p: 0.553
D 1x2 H 1x2: 0.29 p: 0.026
D 1x2 H 1x3: 0.16 p: 0.216
D 1x2 H 1x4: 0.29 p: 0.024
D 1x2 H 2x2: 0.27 p: 0.034
D 1x2 H 2x3: -0.04 p: 0.774
D 1x2 H 2x4: 0.12 p: 0.357
D 1x2 H 3x3: 0.03 p: 0.82
D 1x2 H 3x4: 0.08 p: 0.541
D 1x2 H 4x4: 0.06 p: 0.668
D 1x3 H 1x1: 0.36 p: 0.004
D 1x3 H 1x2: 0.22 p: 0.092
D 1x3 H 1x3: -0.05 p: 0.701
D 1x3 H 1x4: 0.21 p: 0.113
D 1x3 H 2x2: 0.21 p: 0.113
D 1x3 H 2x3: -0.14 p: 0.293
D 1x3 H 2x4: -0.02 p: 0.875
D 1x3 H 3x3: -0.22 p: 0.097
D 1x3 H 3x4: -0.25 p: 0.054
D 1x3 H 4x4: 0.01 p: 0.918
D 2x3 H 1x1: -0.18 p: 0.162
D 2x3 H 1x2: -0.22 p: 0.086
D 2x3 H 1x3: 0.09 p: 0.492
D 2x3 H 1x4: -0.24 p: 0.07
D 2x3 H 2x2: 0.51 p: 0.0
D 2x3 H 2x3: -0.25 p: 0.058
D 2x3 H 2x4: 0.14 p: 0.29
D 2x3 H 3x3: 0.12 p: 0.349
D 2x3 H 3x4: -0.07 p: 0.619
D 2x3 H 4x4: 0.4 p: 0.001


In [273]:
px.line(Desc[0,:,:,0])

In [22]:
posisP = ['Mão', 'Peito', 'Tornozelo']
Qui, Pv, rotulo = [], [], []
for d1 in range(3):
    for d2 in range(d1+1, 3):
        for l1 in range(L):
            for l2 in range(L):
                s1 = np.sign(Desc[d1,21:80,l1,0] - Desc[d1,20:79,l1,0])
                s2 = np.sign(Desc[d2,21:80,l2,0] - Desc[d2,20:79,l2,0])
                tC = np.zeros((2,2), dtype=int)
                tC[0,0] = np.sum(s1==1)
                tC[0,1] = np.sum(s1==-1)
                tC[1,0] = np.sum(s2==1)
                tC[1,1] = np.sum(s2==-1)
                tE = np.zeros((2,2))
                for i in range(2):
                    for j in range(2):
                        tE[i,j] = tC[i,:].sum() * tC[:,j].sum() / tC.sum()
                qui2 = ((tC - tE)**2 / tE).sum()
                pValor = 1 - scipy.stats.chi2.cdf(qui2, df=1)
                Qui.append(qui2)
                Pv.append(pValor)
                rotulo.append(f'P {d1} com {d2} - H {l1+1} com {l2+1}')

In [23]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=Qui, mode='markers', name='Qui-quadrado'))
fig.add_trace(go.Scatter(y=Pv, mode='markers', name='p-valor'))
fig.update_layout(title='Teste de Qui-quadrado para comparação entre domínios', xaxis_title='Par de domínios comparados', yaxis_title='Valor do teste', height=700)
fig.update_xaxes(tickmode='array', tickvals=np.arange(len(rotulo)), ticktext=rotulo)
fig.show()

In [24]:
np.array(rotulo)[np.array(Pv)<0.05]

array([], dtype='<U21')